# 01 — Hello, model

### *"The conversation is an illusion."*

We are going to build a data-analysis agent from nothing. No LangChain, no framework, no magic.
By notebook 07 it will explore messy clinical data, catch its own mistakes, and hand back a
structured, audited answer — and every line of it will be Python you watched get written.

This chapter has exactly one job: **show you what the raw material actually is.**

Everything else in this series is a response to a problem you are about to watch happen.

## 0. A promise, and how to check I'm keeping it

I said **no frameworks**. So before anything else, here is the *entire* dependency list for the
agent:

| | what it does | what it does **not** do |
|---|---|---|
| `openai` | makes an HTTPS POST and parses the JSON back | any agent logic |
| `pandas` | dataframes | any agent logic |
| `pydantic` | validates a dict against a schema | any agent logic |

That's it. No LangChain, no LlamaIndex, no agent library.

You'll also see me import from **`agentlib/`** — that is **not a framework, it's *our* code.**
It's the thing we are building. Every line of it gets written in front of you across these seven
notebooks; by notebook 07 you'll have seen all of it. It's the *destination*, not a dependency.

But I'm not going to start by handing you a helper and saying "trust me." So let's begin with the
rawest possible thing.

## 1. The raw API call. No wrapper, no helper, nothing.

This is the actual HTTP request to Nebius Token Factory, in full. Nothing is hidden.

In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env")

client = OpenAI(
    base_url="https://api.tokenfactory.nebius.com/v1/",   # ← Nebius, not OpenAI
    api_key=os.environ["NEBIUS_API_KEY"],
)

response = client.chat.completions.create(
    model="Qwen/Qwen3-30B-A3B-Instruct-2507",
    messages=[{"role": "user", "content": "In one sentence: what is a p-value?"}],
)

print(response.choices[0].message.content)

A p-value is the probability of obtaining results at least as extreme as the observed results, assuming the null hypothesis is true.


**That is the whole thing.** `base_url` + `api_key` was the entire integration with Nebius — the
`openai` package is just an HTTP client, and Token Factory speaks the same protocol.

Everything else in these seven notebooks is built on top of *that call*. There is no other magic
ingredient. If you understand this cell, you understand the foundation of every agent you have
ever used.

## 1b. Now we wrap it — and here is exactly what the wrapper adds

Calling `client.chat.completions.create(...)` by hand every time is tedious, so we write one
helper. **It adds three things and nothing else:**

| | why |
|---|---|
| **retry** | networks fail. Exponential backoff. |
| **a cost meter** | an agent makes *many* calls. If you can't say what a run cost, you can't make engineering decisions about it. |
| **a disk cache** | same request → same answer, instantly, free, offline. This is why these notebooks run **without an API key** (`set_live(False)`), and — as we'll see in notebook 07 — it's how you write deterministic tests for a non-deterministic system. |

That's `agentlib/llm.py`. **~118 lines. Go read it — it's shorter than this markdown cell.**
It does *not* do anything clever, and I'd rather you verified that than believed me.

In [3]:
import sys
sys.path.insert(0, os.path.abspath(".."))

from agentlib.llm import llm, METER      # our own code. ~118 lines. no magic.
from agentlib import config

print("model :", config.AGENT_MODEL)
print("host  :", config.BASE_URL)

# same call as above, now through our helper
msg = llm([{"role": "user", "content": "In one sentence: what is a p-value?"}])
print("\n", msg.content)

model : Qwen/Qwen3-30B-A3B-Instruct-2507
host  : https://api.tokenfactory.nebius.com/v1/

 A p-value is the probability of obtaining results at least as extreme as the observed data, assuming the null hypothesis is true.


## 2. An LLM call is a function. That's all it is.

You send a list of messages. You get a message back. There is no memory, no state, no session.

It's `f(messages) -> message`. Nothing more.

## 3. The "conversation" is something *you* maintain

The model does not remember the last message. It cannot. Each call is fresh — the model sees
the entire list of messages, from scratch, every single time.

When you chat with an LLM and it "remembers" what you said, that is **the client resending the
whole history on every turn.** The illusion of memory is just a growing list.

Watch. First, ask a follow-up *without* the history:

In [4]:
# No history. The model has no idea what "it" refers to.
orphan = llm([{"role": "user", "content": "What's a common mistake people make with it?"}])
print("WITHOUT history:\n", orphan.content[:300])

WITHOUT history:
 A common mistake people make with "it" is using it incorrectly as a pronoun when the antecedent (the noun it refers to) is unclear or ambiguous. For example:

❌ *"When I saw the dog, it was barking."*

This sentence is unclear—does "it" refer to the dog or the person who saw the dog? The meaning dep


In [5]:
# Now with the history — we resend everything, and *we* are the ones doing the remembering.
conversation = [
    {"role": "user", "content": "In one sentence: what is a p-value?"},
    {"role": "assistant", "content": msg.content},
    {"role": "user", "content": "What's a common mistake people make with it?"},
]
withctx = llm(conversation)
print("WITH history:\n", withctx.content[:300])

WITH history:
 A common mistake is interpreting the p-value as the probability that the null hypothesis is true, when in fact it only measures the probability of the data (or more extreme data) given that the null hypothesis is true.


> ### 💡 The first thing to internalise
>
> **You own the context.** Nothing goes into the model's head that you did not put there.
>
> That sounds obvious. It is the single most important fact in this entire series — because
> every failure we hit from here on will be, in some form, *the wrong thing being in the
> context, or the right thing missing from it.* Designing an agent is mostly designing what
> the model gets to see.

---
## 4. Now the problem. Ask it to do our actual job.

We have a real CSV on disk: 344 penguins. Let's ask about it.

In [7]:
import pandas as pd

penguins = pd.read_csv("data/penguins.csv")
print(penguins.head(3).to_string())
print(f"\n{len(penguins)} rows")

  species     island  bill_length_mm  bill_depth_mm  flipper_length_mm  body_mass_g     sex
0  Adelie  Torgersen            39.1           18.7              181.0       3750.0    Male
1  Adelie  Torgersen            39.5           17.4              186.0       3800.0  Female
2  Adelie  Torgersen            40.3           18.0              195.0       3250.0  Female

344 rows


The model **cannot see this file.** It's on my disk. The model is on a GPU in another country.

So what happens if we just... ask it anyway?

In [6]:
naive = llm([{"role": "user", "content":
              "The file data/penguins.csv contains the Palmer Penguins dataset. "
              "What is the mean body mass in grams? Reply with just the number."}])
print("MODEL SAYS :", naive.content.strip()[:80])
print("THE TRUTH  :", penguins["body_mass_g"].mean())

MODEL SAYS : 4201.754594594595
THE TRUTH  : 4201.754385964912


### Wait. That's... almost exactly right?

It didn't refuse. It gave a number, and the number is *correct to several decimal places.*

**It did not compute that.** It cannot compute anything; it never saw the file.

It **remembered** it. Palmer Penguins is one of the most-used teaching datasets on the
internet. Its mean body mass is sitting in the model's weights, memorised from a thousand
tutorials. Look closely at the last digits though — they don't quite match. It's not recalling
a fact, it's *reconstructing* one, and the reconstruction is subtly wrong.

> ### 🚨 This is worse than a hallucination, not better.
>
> A model that confidently invents numbers is dangerous. A model that confidently invents
> numbers **and happens to be right on the datasets you test it with** is *far* more dangerous
> — because it will pass your evaluation and then fail on real data.
>
> This is exactly why GeneBench-Pro builds its benchmark from *simulated* data: analysis
> problems are "specifically chosen so they do not recapitulate well-known textbook examples
> or papers, so as to avoid the risk of benchmarking against memorized solutions."

So let's give it a dataset it *cannot* have memorised. This one was generated on my laptop
this week from a random seed. It has never existed before, anywhere.

In [8]:
trial = pd.read_csv("data/trial.csv")
print(f"{len(trial)} rows — synthetic clinical trial data, generated from seed 20260701")
print(trial[["patient_id", "arm", "severity", "biomarker_baseline"]].head(3).to_string(index=False))

848 rows — synthetic clinical trial data, generated from seed 20260701
patient_id       arm severity  biomarker_baseline
      P001 treatment     mild                34.5
      P002 treatment   severe                62.8
      P003   control moderate                53.0


In [9]:
unseen = llm([{"role": "user", "content":
               "The file data/trial.csv contains a clinical trial dataset with a column "
               "biomarker_baseline. What is its mean value? Reply with just the number."}])
print("MODEL SAYS :", unseen.content.strip()[:80])
print("THE TRUTH  :", round(trial["biomarker_baseline"].mean(), 2))

MODEL SAYS : 12.34
THE TRUTH  : -64.0


> ### 🚨 *There* it is.
>
> No memorised answer to fall back on, so it **invented one.** A confident, plausible,
> specific number with believable units and magnitude. It did not say *"I cannot see that
> file."* It just... answered.
>
> In data analysis this is the worst failure mode there is: not an error, but **a wrong answer
> that looks exactly like a right one.** If this number landed in a slide, nobody would blink.

*(And note the true value is bizarre — a negative mean for a biomarker. Hold that thought.
That's a landmine we'll step on properly in notebook 04.)*

## 5. "Fine — I'll paste the data in."

The obvious fix. Give it the rows and let it compute. Let's paste in 40 of them and ask for
something a little harder than a mean.

In [19]:
sample = penguins.dropna().head(40)
pasted = llm([{"role": "user", "content":
               f"Here is a table of penguins:\n\n{sample.to_csv(index=False)}\n\n"
               "What is the standard deviation of body_mass_g? Reply with just the number."}])

truth = sample["body_mass_g"].std()
print("MODEL SAYS :", pasted.content.strip()[:80])
print("THE TRUTH  :", round(truth, 2))

MODEL SAYS : 456.05
THE TRUTH  : 439.23


It can see every single number, and it *still* gets it wrong (or, if it happens to get close,
it got close by luck — run it again and watch it move).

This is not a knowledge problem. The model knows the formula for standard deviation perfectly.
It's an *arithmetic* problem: the model is predicting tokens, not executing operations.

And pasting doesn't scale anyway. 40 rows fit. Our real dataset — the one in notebook 04 — has
848. A production table has 200 million. You cannot paste a database into a prompt.

---
## 6. The insight that shapes everything after this

The model is a **bad calculator**.

But watch this:

In [20]:
code = llm([{"role": "user", "content":
             "Write Python (pandas) to compute the standard deviation of the body_mass_g column "
             "in data/penguins.csv. Just the code, no explanation."}])
print(code.content)

import pandas as pd

df = pd.read_csv('data/penguins.csv')
std_dev = df['body_mass_g'].std()


That code is **perfect.** It would give the exactly correct answer, on 344 rows or 344 million.

> # 🎯 The whole series in one line
>
> ## The model is a bad calculator and a great programmer.
> ## So stop asking it for answers. Ask it for **code**, and run the code yourself.

That is what a data-analysis agent *is*. Everything from here is working out how to do that
reliably — because as we're about to find out, "run the code it writes" is where the real
problems start, not where they end.

---
## 7. What the wrapper was quietly doing for us

Before we go further, two small pieces of infrastructure that will run through every notebook.
They're already inside `llm()` — worth knowing they're there.

**A cost meter.** An agent makes *many* calls. If you can't say what a run costs, you can't
make engineering decisions about it. So every call is metered.

**A cache.** Every response is saved to disk, keyed by a hash of the request. Same request →
same answer, instantly, free, offline. That means:
- these notebooks run without an API key (set `LIVE = False`),
- the walkthrough can't be sabotaged by a rate limit or a bad day from the model,
- and — as we'll see in notebook 07 — it's how you write *deterministic tests for a
  non-deterministic system*.

In [23]:
print("this notebook so far:", METER)

this notebook so far: 15 calls (2 billed, 13 cached) | 12,736 in + 350 out = 13,086 tokens | $0.0002


Under a cent. Good — because we're going to make a *lot* of calls.

---
# Where we are

| | |
|---|---|
| **We learned** | An LLM call is `f(messages) → message`. You own the context entirely. |
| **We saw it fail** | On a *famous* dataset it recited a memorised answer (dangerous — it looks like competence). On an *unseen* one it invented a confident number. Given data it could see, it did the arithmetic wrong. |
| **The way out** | It writes *excellent* code. Let it write code; we'll run it. |

### 🔜 But there's a gap.

We got code back — as a **string, in prose**, wrapped in markdown fences. To use it we'd have to
regex it out of the text and hope the model didn't add commentary. That's brittle and horrible.

There is a proper protocol for "the model wants something done." It's called **tool calling**,
and it is nowhere near as magical as it sounds.

**→ `02_tool_calling.ipynb`**